# Inference Benchmark

In [1]:
import json
from pathlib import Path

import yaml

from edu_eval.inference.benchmark import benchmark_generation
from edu_eval.models.loader import ModelLoader
from edu_eval.models.registry import ModelRegistry

In [2]:
PROJECT_ROOT = Path.cwd().parent
with (PROJECT_ROOT / "configs" / "inference.yaml").open("r", encoding="utf-8") as f:
    max_tokens = int(yaml.safe_load(f)["inference"]["max_new_tokens"])

registry = ModelRegistry.from_yaml(PROJECT_ROOT / "configs" / "models.yaml")
spec = registry.get("smoke_qwen")
tokenizer, model, device = ModelLoader.load(spec)
print(f"{device} {next(model.parameters()).dtype}")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

cuda torch.bfloat16


In [3]:
prompts = [
    "Jelaskan fotosintesis untuk siswa kelas 6 SD.",
    "Jelaskan rantai makanan untuk siswa kelas 7 SMP.",
    "Apa itu kalimat efektif? Jelaskan singkat.",
    "Sebutkan sila Pancasila dan artinya secara singkat.",
]
result = benchmark_generation(model=model, tokenizer=tokenizer, device=device, prompts=prompts, max_new_tokens=max_tokens)
OUT_DIR = PROJECT_ROOT / "results" / "inference"
OUT_DIR.mkdir(parents=True, exist_ok=True)
with (OUT_DIR / "smoke_bf16_reference.json").open("w", encoding="utf-8") as f:
    json.dump({"model_id": "smoke_qwen", "precision": "bf16", **result}, f, ensure_ascii=False, indent=2)
print(f"{result['tokens_per_second']:.1f} tok/s ttft={result['mean_ttft_seconds']:.3f}s vram={result['peak_vram_gb']:.2f}GB")

67.7 tok/s ttft=0.025s vram=0.93GB
